In [2]:
import pandas as pd

def add_dma(df: pd.DataFrame, window: int = 50) -> pd.DataFrame:
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.drop_duplicates(["Ticker", "Date"], keep="last")
    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)  # sort is essential
    df[f"DMA{window}"] = (
        df.groupby("Ticker")["Close"]
          .transform(lambda s: s.rolling(window, min_periods=window).mean())
    )
    return df

df = pd.read_parquet(r"C:\Trading_algorithm\Swing Trading\nifty_500_stocks_data.parquet")
df = add_dma(df)

print(df.dtypes)
print(df.head(3))
print(df[df.Ticker == df.Ticker.iloc[0]].iloc[45:55][["Date", "Ticker", "Close", "DMA50"]])

Date      datetime64[ns]
Ticker            object
Open             float64
High             float64
Low              float64
Close            float64
Volume             int64
DMA50            float64
dtype: object
        Date  Ticker        Open        High         Low       Close   Volume  \
0 2019-09-19  360ONE  244.034802  256.236542  244.034802  256.236542  7109860   
1 2019-09-20  360ONE  268.034927  269.043335  268.034927  269.043335  2760648   
2 2019-09-23  360ONE  282.495483  282.495483  282.495483  282.495483    34096   

   DMA50  
0    NaN  
1    NaN  
2    NaN  
         Date  Ticker       Close       DMA50
45 2019-11-27  360ONE  263.470947         NaN
46 2019-11-28  360ONE  272.291321         NaN
47 2019-11-29  360ONE  266.519440         NaN
48 2019-12-02  360ONE  260.605408         NaN
49 2019-12-03  360ONE  257.069153  256.000833
50 2019-12-04  360ONE  253.248322  255.941069
51 2019-12-05  360ONE  253.725906  255.634720
52 2019-12-06  360ONE  248.614670  254.957104
53 

In [4]:
def add_position(df: pd.DataFrame, window: int = 50) -> pd.DataFrame:
    df = df.copy()
    dma = f"DMA{window}"
    df["Position"] = None                      # stays empty where DMA isn't available yet
    has_dma = df[dma].notna()
    df.loc[has_dma & (df["Close"] > df[dma]), "Position"] = "Above"
    df.loc[has_dma & (df["Close"] <= df[dma]), "Position"] = "Below"
    return df

df = add_position(df)
print(df[df.Ticker == df.Ticker.iloc[0]].tail(5)[["Date", "Close", "DMA50", "Position"]])

           Date        Close        DMA50 Position
1761 2026-11-03  1065.000000  1075.537228    Below
1762 2026-11-04  1068.699951  1073.103004    Below
1763 2026-11-05  1140.900024  1072.295463    Above
1764 2026-11-06  1145.000000  1071.168815    Above
1765 2026-11-09  1138.000000  1070.156334    Above


In [10]:
def daily_breadth(df: pd.DataFrame) -> pd.DataFrame:
    counts = (
        df.dropna(subset=["Position"])
          .groupby(["Date", "Position"]).size()
          .unstack(fill_value=0)
          .reindex(columns=["Above", "Below"], fill_value=0)
    )
    counts["Total"] = counts["Above"] + counts["Below"]
    counts["Pct_Above"] = 100 * counts["Above"] / counts["Total"]
    counts["Pct_Below"] = 100 * counts["Below"] / counts["Total"]
    counts = counts.reset_index()
    counts["Date"] = pd.to_datetime(counts["Date"])
    return counts


breadth = daily_breadth(df)

In [11]:
display(breadth)

Position,Date,Above,Below,Total,Pct_Above,Pct_Below
0,1996-03-08,15,2,17,88.235294,11.764706
1,1996-03-11,15,3,18,83.333333,16.666667
2,1996-03-12,13,6,19,68.421053,31.578947
3,1996-03-13,15,4,19,78.947368,21.052632
4,1996-03-14,16,3,19,84.210526,15.789474
...,...,...,...,...,...,...
7694,2026-11-03,143,354,497,28.772636,71.227364
7695,2026-11-04,187,310,497,37.625755,62.374245
7696,2026-11-05,236,261,497,47.484909,52.515091
7697,2026-11-06,228,269,497,45.875252,54.124748


In [30]:
breadth.to_csv(r"C:\Trading_algorithm\Swing Trading\dma_breadth.csv", index=True)

In [29]:
import plotly.express as px
from IPython.display import HTML

breadth_last_5_years = breadth[
    breadth["Date"] >= (breadth["Date"].max() - pd.DateOffset(years=5))
].sort_values("Date").copy()

fig = px.line(
    breadth_last_5_years,
    x="Date",
    y="Pct_Above",
    title="% of Stocks Above 50-Day Moving Average",
    labels={"Date": "Date", "Pct_Above": "% Above 50 DMA"},
    markers=True,
    template="plotly_white",
)

fig.update_layout(
    width=1400,
    height=600,
    title_x=0.5,
    xaxis_title="Date",
    yaxis_title="% Above 50 DMA",
    yaxis=dict(range=[0, 100]),
)

html_path = r"C:\Trading_algorithm\Swing Trading\breadth_50dma_interactive.html"
fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
with open(html_path, "r", encoding="utf-8") as f:
    display(HTML(f.read()))

print(breadth_last_5_years[["Date", "Above", "Below", "Pct_Above", "Pct_Below"]].tail(10))

Position       Date  Above  Below  Pct_Above  Pct_Below
7689     2026-10-27     90    407  18.108652  81.891348
7690     2026-10-28     96    401  19.315895  80.684105
7691     2026-10-29    103    394  20.724346  79.275654
7692     2026-10-30     95    402  19.114688  80.885312
7693     2026-11-02    119    378  23.943662  76.056338
7694     2026-11-03    143    354  28.772636  71.227364
7695     2026-11-04    187    310  37.625755  62.374245
7696     2026-11-05    236    261  47.484909  52.515091
7697     2026-11-06    228    269  45.875252  54.124748
7698     2026-11-09    257    240  51.710262  48.289738
